# Run SOLWEIG for MRT and UTCI

This notebook uses the GeoTIFFs generated previously:

```text
/media/harshin/data_drive/solweig/tif/
├── dem.tif
├── dsm.tif
└── cdsm.tif
```

Run configuration taken **only** from the supplied `start.sh` comparison-run launcher:

- Date: **2025-07-06**
- Location: **25.7560, -80.3770**
- Time zone: **America/New_York**
- Time step: **10 minutes**
- Assumed run window: **00:00–23:50 local time** (144 timesteps)
- Fallback RH: **70%**
- Fallback wind speed: **3.1 m/s**

The script itself is not used for geometry, physics, radiation, or any other SOLWEIG setting.

## Weather choice

SOLWEIG requires air temperature, relative humidity, global solar radiation, and preferably wind speed.

This notebook supports:

1. **CSV mode** — recommended for comparison using the actual weather series.
2. **EPW mode** — fallback using a downloaded typical-year EPW file, interpolated to 10-minute intervals.

EPW results are not actual observations for July 6, 2025 and should not be presented as such.


> **Comparison-forcing update:** air temperature, relative humidity, and wind speed are now read from the same `weather.csv` used by the custom `route_utci` workflow. EPW is used only for global shortwave radiation because the comparison CSV does not contain radiation.


## 1. Install/check packages

In [ ]:
%pip install -q solweig pandas rasterio matplotlib

Restart the kernel once if packages were newly installed, then continue.

## 2. Imports and run configuration

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import inspect

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import solweig

print("SOLWEIG:", getattr(solweig, "__version__", "version not exposed"))
print("Module:", solweig.__file__)

In [ ]:
BASE = Path("/media/harshin/data_drive/solweig")
TIF_DIR = BASE / "tif"
CACHE_DIR = BASE / "solweig_cache"
OUTPUT_DIR = BASE / "solweig_output_same_met_2025-07-06"
WEATHER_DIR = BASE / "weather"

DSM_PATH = TIF_DIR / "dsm.tif"
DEM_PATH = TIF_DIR / "dem.tif"
CDSM_PATH = TIF_DIR / "cdsm.tif"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEATHER_DIR.mkdir(parents=True, exist_ok=True)

RUN_DATE = "2025-07-06"
LATITUDE = 25.7560
LONGITUDE = -80.3770
TIMEZONE = "America/New_York"
UTC_OFFSET = -4
DT_MINUTES = 10

START_LOCAL = pd.Timestamp(f"{RUN_DATE} 00:00:00", tz=TIMEZONE)
END_LOCAL = pd.Timestamp(f"{RUN_DATE} 23:50:00", tz=TIMEZONE)
TARGET_TIMES = pd.date_range(
    START_LOCAL,
    END_LOCAL,
    freq=f"{DT_MINUTES}min",
)

FALLBACK_RH_PCT = 70.0
FALLBACK_WIND_MS = 3.1

# Comparison mode:
# - Ta, RH, and wind come from the same weather.csv used by route_utci.
# - Global shortwave radiation comes from the EPW/TMY file because the
#   uploaded comparison CSV does not contain a radiation column.
WEATHER_MODE = "same_met_epw_radiation"

WEATHER_CSV = WEATHER_DIR / "weather.csv"
EPW_PATH = WEATHER_DIR / "miami.epw"

print("Weather mode:", WEATHER_MODE)
print("Timesteps:", len(TARGET_TIMES))
print("Start:", TARGET_TIMES[0])
print("End:", TARGET_TIMES[-1])
print("Comparison weather CSV:", WEATHER_CSV)
print("EPW radiation source:", EPW_PATH)

for p in [DSM_PATH, DEM_PATH, CDSM_PATH]:
    print(p.name, p.exists(), p)


## 3. Validate raster alignment and CRS

In [ ]:
def raster_info(path):
    with rasterio.open(path) as src:
        arr = src.read(1, masked=True)
        return {
            "path": path,
            "shape": src.shape,
            "crs": src.crs,
            "transform": src.transform,
            "resolution": src.res,
            "bounds": src.bounds,
            "min": float(arr.min()),
            "max": float(arr.max()),
        }

infos = [raster_info(p) for p in [DSM_PATH, DEM_PATH, CDSM_PATH]]

for info in infos:
    print("\n", info["path"].name)
    for key in ["shape", "crs", "resolution", "bounds", "min", "max"]:
        print(f"{key}: {info[key]}")

assert infos[0]["shape"] == infos[1]["shape"] == infos[2]["shape"]
assert infos[0]["crs"] == infos[1]["crs"] == infos[2]["crs"]
assert infos[0]["transform"] == infos[1]["transform"] == infos[2]["transform"]
assert str(infos[0]["crs"]) == "EPSG:6346", f"Unexpected CRS: {infos[0]['crs']}"

print("\nRaster alignment and CRS checks passed.")

## 4. Prepare SOLWEIG surface data

In [ ]:
print(inspect.signature(solweig.SurfaceData.prepare))

In [ ]:
# This computes and caches walls and sky-view factors.
# The first run can take substantially longer than subsequent runs.
surface = solweig.SurfaceData.prepare(
    dsm=str(DSM_PATH),
    dem=str(DEM_PATH),
    cdsm=str(CDSM_PATH),
    working_dir=str(CACHE_DIR),
)

print("Surface preparation complete.")

## 5A. CSV weather loader

Expected columns are flexible. The notebook recognizes common names for:

- datetime or date/time
- air temperature
- relative humidity
- wind speed
- global horizontal radiation

For the strongest comparison, provide the same meteorological series used by the other simulation.


In [ ]:
def find_column(df, candidates, required=True):
    lookup = {
        str(column).strip().lower(): column
        for column in df.columns
    }

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    if required:
        raise KeyError(
            f"None of these columns were found: {candidates}"
        )

    return None


def load_comparison_met_csv(path):
    """
    Load the hourly comparison meteorology used by route_utci.

    Expected columns:
        hour, air_temp_C, rh_pct, wind_ms

    Hour 24 is interpreted as midnight at the start of the following day.
    The hourly values are interpolated to TARGET_TIMES.
    """
    df = pd.read_csv(path)

    hour_col = find_column(df, ["hour"])
    ta_col = find_column(
        df,
        ["air_temp_c", "ta", "temperature_c", "temp_c"],
    )
    rh_col = find_column(
        df,
        ["rh_pct", "rh", "relative_humidity"],
    )
    wind_col = find_column(
        df,
        ["wind_ms", "wind_speed_ms", "wind_speed", "ws"],
    )

    hour = pd.to_numeric(df[hour_col], errors="coerce")
    ta = pd.to_numeric(df[ta_col], errors="coerce")
    rh = pd.to_numeric(df[rh_col], errors="coerce")
    wind = pd.to_numeric(df[wind_col], errors="coerce")

    met = pd.DataFrame({
        "hour": hour,
        "ta": ta,
        "rh": rh,
        "wind": wind,
    }).dropna()

    if met.empty:
        raise ValueError(
            "Comparison weather CSV contains no valid records."
        )

    if not met["hour"].between(0, 24).all():
        raise ValueError(
            "The hour column must contain values between 0 and 24."
        )

    run_midnight = pd.Timestamp(
        f"{RUN_DATE} 00:00:00",
        tz=TIMEZONE,
    )

    met["datetime"] = (
        run_midnight
        + pd.to_timedelta(met["hour"], unit="h")
    )

    met = (
        met
        .set_index("datetime")
        .sort_index()
    )

    if met.index.duplicated().any():
        met = met[~met.index.duplicated(keep="first")]

    combined_index = met.index.union(TARGET_TIMES).sort_values()

    met_target = (
        met[["ta", "rh", "wind"]]
        .reindex(combined_index)
        .interpolate(method="time")
        .reindex(TARGET_TIMES)
    )

    if met_target.isna().any().any():
        raise ValueError(
            "Comparison weather CSV does not cover the complete "
            "SOLWEIG run window after interpolation."
        )

    met_target["rh"] = met_target["rh"].clip(0.0, 100.0)
    met_target["wind"] = met_target["wind"].clip(lower=0.0)

    return met_target


## 5B. EPW fallback loader

In [ ]:
def load_epw_radiation_to_target():
    """
    Load a representative July 6 from the EPW/TMY file and interpolate
    global shortwave radiation to TARGET_TIMES.

    Only radiation is retained from EPW. Air temperature, RH, and wind
    are supplied by the comparison CSV.
    """
    epw_path = EPW_PATH

    if not epw_path.exists():
        print("Downloading EPW...")
        downloaded = solweig.download_epw(
            latitude=LATITUDE,
            longitude=LONGITUDE,
            output_path=str(epw_path),
        )
        epw_path = (
            Path(downloaded)
            if downloaded is not None
            else epw_path
        )

    epw_weather = solweig.Weather.from_epw(
        str(epw_path)
    )

    rows = []

    for weather in epw_weather:
        timestamp = pd.Timestamp(weather.datetime)

        if timestamp.tzinfo is None:
            timestamp = timestamp.tz_localize(TIMEZONE)
        else:
            timestamp = timestamp.tz_convert(TIMEZONE)

        if timestamp.month == 7 and timestamp.day == 6:
            rows.append({
                "source_datetime": timestamp,
                "global_rad": float(weather.global_rad),
            })

    if not rows:
        raise RuntimeError(
            "No July 6 records were found in the EPW file."
        )

    radiation = (
        pd.DataFrame(rows)
        .set_index("source_datetime")
        .sort_index()
    )

    run_year = pd.Timestamp(RUN_DATE).year

    radiation.index = pd.DatetimeIndex([
        pd.Timestamp(
            year=run_year,
            month=timestamp.month,
            day=timestamp.day,
            hour=timestamp.hour,
            minute=timestamp.minute,
            second=timestamp.second,
            tz=TIMEZONE,
        )
        for timestamp in radiation.index
    ])

    radiation = radiation[
        ~radiation.index.duplicated(keep="first")
    ]

    # Add the next-midnight endpoint when needed so 23:50 can be
    # interpolated rather than extrapolated.
    next_midnight = (
        pd.Timestamp(
            f"{RUN_DATE} 00:00:00",
            tz=TIMEZONE,
        )
        + pd.Timedelta(days=1)
    )

    if next_midnight not in radiation.index:
        last_value = float(
            radiation["global_rad"].iloc[-1]
        )
        radiation.loc[next_midnight, "global_rad"] = max(
            last_value,
            0.0,
        )

    combined_index = (
        radiation.index
        .union(TARGET_TIMES)
        .sort_values()
    )

    radiation_target = (
        radiation[["global_rad"]]
        .reindex(combined_index)
        .interpolate(method="time")
        .reindex(TARGET_TIMES)
    )

    radiation_target["global_rad"] = (
        radiation_target["global_rad"]
        .clip(lower=0.0)
    )

    if radiation_target.isna().any().any():
        raise ValueError(
            "EPW radiation interpolation left missing values."
        )

    return radiation_target, epw_path


## 6. Build the 10-minute weather series

In [ ]:
if WEATHER_MODE.lower() == "same_met_epw_radiation":
    if not WEATHER_CSV.exists():
        raise FileNotFoundError(
            "The comparison meteorology file is missing: "
            f"{WEATHER_CSV}"
        )

    comparison_met = load_comparison_met_csv(
        WEATHER_CSV
    )

    epw_radiation, epw_source = (
        load_epw_radiation_to_target()
    )

    weather_df = comparison_met.join(
        epw_radiation,
        how="inner",
    )

    weather_source = {
        "ta_rh_wind": str(WEATHER_CSV),
        "global_radiation": str(epw_source),
    }

elif WEATHER_MODE.lower() == "epw":
    # Retained as an optional fallback mode.
    weather_df, epw_source = epw_weather_to_target()
    weather_source = {"all_variables": str(epw_source)}

else:
    raise ValueError(
        "WEATHER_MODE must be "
        "'same_met_epw_radiation' or 'epw'."
    )

required_weather_columns = [
    "ta",
    "rh",
    "wind",
    "global_rad",
]

missing_columns = [
    column
    for column in required_weather_columns
    if column not in weather_df.columns
]

if missing_columns:
    raise ValueError(
        "Weather forcing is missing columns: "
        f"{missing_columns}"
    )

if weather_df[required_weather_columns].isna().any().any():
    raise ValueError(
        "Weather forcing contains missing values."
    )

print("Weather sources:")
for key, value in weather_source.items():
    print(f"  {key}: {value}")

print("\nFirst weather rows:")
display(weather_df.head())

print("\nMidday comparison weather:")
display(
    weather_df.between_time("11:00", "15:00")
)

print("\nWeather statistics:")
display(weather_df.describe())


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(weather_df.index, weather_df["ta"])
plt.ylabel("Air temperature, °C")
plt.title("SOLWEIG weather forcing: air temperature")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(weather_df.index, weather_df["global_rad"])
plt.ylabel("Global radiation, W/m²")
plt.title("SOLWEIG weather forcing: global shortwave radiation")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Verify identical meteorological forcing

This check confirms that SOLWEIG receives the same air temperature, relative humidity, and wind speed used by the custom `route_utci` workflow. Radiation remains EPW-derived because the comparison CSV does not include radiation.


In [ ]:
comparison_source_hourly = pd.read_csv(WEATHER_CSV)

check_hours = [0, 6, 12, 13, 18, 23]

verification_rows = []

for hour in check_hours:
    source_row = comparison_source_hourly.loc[
        comparison_source_hourly["hour"] == hour
    ].iloc[0]

    target_time = pd.Timestamp(
        f"{RUN_DATE} {hour:02d}:00:00",
        tz=TIMEZONE,
    )

    solweig_row = weather_df.loc[target_time]

    verification_rows.append({
        "hour": hour,
        "csv_ta_c": float(source_row["air_temp_C"]),
        "solweig_ta_c": float(solweig_row["ta"]),
        "csv_rh_pct": float(source_row["rh_pct"]),
        "solweig_rh_pct": float(solweig_row["rh"]),
        "csv_wind_ms": float(source_row["wind_ms"]),
        "solweig_wind_ms": float(solweig_row["wind"]),
        "epw_global_rad_wm2": float(
            solweig_row["global_rad"]
        ),
    })

forcing_verification = pd.DataFrame(
    verification_rows
)

display(forcing_verification)

for variable_pair in [
    ("csv_ta_c", "solweig_ta_c"),
    ("csv_rh_pct", "solweig_rh_pct"),
    ("csv_wind_ms", "solweig_wind_ms"),
]:
    left, right = variable_pair
    if not np.allclose(
        forcing_verification[left],
        forcing_verification[right],
        atol=1e-10,
    ):
        raise AssertionError(
            f"Forcing mismatch between {left} and {right}."
        )

print(
    "Verified: SOLWEIG Ta, RH, and wind match "
    "the comparison CSV at all checked hours."
)


## 7. Convert weather table to SOLWEIG Weather objects

In [ ]:
print(inspect.signature(solweig.Weather))

In [ ]:
weather_list = []

weather_signature = inspect.signature(solweig.Weather)
weather_parameters = set(weather_signature.parameters)

for timestamp, row in weather_df.iterrows():
    kwargs = {
        "datetime": timestamp.to_pydatetime().replace(tzinfo=None),
        "ta": float(row["ta"]),
        "rh": float(row["rh"]),
        "global_rad": float(row["global_rad"]),
    }

    if "wind_speed" in weather_parameters:
        kwargs["wind_speed"] = float(row["wind"])
    elif "wind" in weather_parameters:
        kwargs["wind"] = float(row["wind"])
    elif "ws" in weather_parameters:
        kwargs["ws"] = float(row["wind"])

    weather_list.append(solweig.Weather(**kwargs))

print("Weather objects:", len(weather_list))
print(weather_list[0])
print(weather_list[-1])

## 8. Define location and validate inputs

In [ ]:
location = solweig.Location(
    latitude=LATITUDE,
    longitude=LONGITUDE,
    utc_offset=UTC_OFFSET,
)

warnings = solweig.validate_inputs(surface, location, weather_list)

if warnings:
    print("Validation warnings:")
    for warning in warnings:
        print("-", warning)
else:
    print("No validation warnings.")

## 9. Run SOLWEIG

In [ ]:
print("Compute backend:", solweig.get_compute_backend())
print("GPU available:", solweig.is_gpu_available())

In [ ]:
summary = solweig.calculate(
    surface=surface,
    weather=weather_list,
    location=location,
    output_dir=str(OUTPUT_DIR),
    outputs=["tmrt", "utci", "shadow"],
)

print(summary.report())

## 10. Inspect summary maps

In [ ]:
def show_grid(grid, title, label):
    plt.figure(figsize=(12, 8))
    image = plt.imshow(grid)
    plt.colorbar(image, label=label)
    plt.title(title)
    plt.xlabel("Raster column")
    plt.ylabel("Raster row")
    plt.tight_layout()
    plt.show()


show_grid(summary.tmrt_mean, "Daily mean MRT", "Tmrt, °C")
show_grid(summary.tmrt_max, "Daily maximum MRT", "Tmrt, °C")
show_grid(summary.utci_mean, "Daily mean UTCI", "UTCI, °C")
show_grid(summary.utci_max, "Daily maximum UTCI", "UTCI, °C")

## 11. Confirm output files and metadata

In [ ]:
print("Output directory:", OUTPUT_DIR)

for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR))

metadata_path = OUTPUT_DIR / "run_metadata.json"
if metadata_path.exists():
    metadata = solweig.load_run_metadata(str(metadata_path))
    print("\nRecorded SOLWEIG version:", metadata.get("solweig_version"))
    print("Recorded location:", metadata.get("location"))
    print("Recorded timeseries:", metadata.get("timeseries"))

## Interpretation note

This notebook will generate MRT and UTCI GeoTIFFs for every 10-minute timestep, plus summary rasters.

For a controlled comparison against your other simulation, use the same:

- air-temperature series;
- RH series;
- wind-speed series;
- global shortwave-radiation series;
- time convention and UTC offset.

Changing only the geometry/model while keeping the forcing identical makes the comparison interpretable.
